In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import mlflow

from transformers import AutoModel
from torch.utils.data import DataLoader
from datasets import load_dataset

from src.model import dinosplus_classfier
from src.training import train
from src.data import setdata

mlflow.set_tracking_uri("sqlite:///mlruns.db")

data = load_dataset("zh-plus/tiny-imagenet")

train_ds = setdata(data["train"])
test_ds  = setdata(data["valid"])

train_loader = DataLoader(
    train_ds,
    batch_size=128,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

test_loader = DataLoader(
    test_ds,
    batch_size=128,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

backbone = AutoModel.from_pretrained("facebook/deit-small-patch16-224")
vit = dinosplus_classfier(backbone, num=200)

# ✅ full fine-tuning
for p in vit.parameters():
    p.requires_grad = True

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
if device == "cuda":
    print(torch.cuda.get_device_name(0))

optimizer = optim.Adam(vit.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler("cuda" if device == "cuda" else "cpu")

train(
    vit,
    train_loader,
    test_loader,
    100,
    criterion,
    scaler,
    device,
    optimizer,
    "deit_small_baseline_full_finetuning",
    "/home/hyuksu/deep-learning-study/outputs/deit_small_baseline_full.pth",
    patience=3,
    gate_lambda=0.0,
)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: facebook/deit-small-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


cuda
NVIDIA GeForce RTX 5060 Ti


deit_small_baseline_full_finetuning:   0%|          | 0/100 [00:00<?, ?it/s]

Test Loss=0.7585, Test Accuracy=0.8096
Epoch 000 | Train Loss=1.4435 | Train Acc=0.7195 | Test Loss=0.7585 | Test Acc=0.8096 | Gate Loss=0.00000
Test Loss=0.7432, Test Accuracy=0.8145
Epoch 001 | Train Loss=0.4623 | Train Acc=0.8815 | Test Loss=0.7432 | Test Acc=0.8145 | Gate Loss=0.00000
Test Loss=0.7529, Test Accuracy=0.8210
Epoch 002 | Train Loss=0.2519 | Train Acc=0.9325 | Test Loss=0.7529 | Test Acc=0.8210 | Gate Loss=0.00000
Test Loss=0.8042, Test Accuracy=0.8190
Epoch 003 | Train Loss=0.1563 | Train Acc=0.9564 | Test Loss=0.8042 | Test Acc=0.8190 | Gate Loss=0.00000
Test Loss=0.8821, Test Accuracy=0.8042
Epoch 004 | Train Loss=0.1183 | Train Acc=0.9661 | Test Loss=0.8821 | Test Acc=0.8042 | Gate Loss=0.00000


2026/04/09 00:58:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Test Loss=0.9068, Test Accuracy=0.8103
Epoch 005 | Train Loss=0.0964 | Train Acc=0.9720 | Test Loss=0.9068 | Test Acc=0.8103 | Gate Loss=0.00000
Early Stopping Triggered at epoch 5


2026/04/09 00:58:28 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


최고 성능 모델이 MLflow에 저장되었습니다.


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import mlflow

from transformers import AutoModel
from torch.utils.data import DataLoader
from datasets import load_dataset

from src.model import dinosplus_classfier
from src.training import train
from src.data import setdata
from src.re_make_Gate import inject_gating

mlflow.set_tracking_uri("sqlite:///mlruns.db")

data = load_dataset("zh-plus/tiny-imagenet")

train_ds = setdata(data["train"])
test_ds  = setdata(data["valid"])

train_loader = DataLoader(
    train_ds,
    batch_size=128,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

test_loader = DataLoader(
    test_ds,
    batch_size=128,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

backbone = AutoModel.from_pretrained("facebook/deit-small-patch16-224")
backbone, hooks = inject_gating(
    backbone,
    keep_cls_ungated=True,
    target_layers=None   # 마지막 1개 레이어 추천
)

vit = dinosplus_classfier(backbone, num=200)

# ✅ full fine-tuning
for p in vit.parameters():
    p.requires_grad = True

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
if device == "cuda":
    print(torch.cuda.get_device_name(0))

optimizer = optim.Adam(vit.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler("cuda" if device == "cuda" else "cpu")

train(
    vit,
    train_loader,
    test_loader,
    100,
    criterion,
    scaler,
    device,
    optimizer,
    "deit_small_gated_full_finetuning",
    "/home/hyuksu/deep-learning-study/outputs/deit_small_gated_full.pth",
    patience=3,
    gate_lambda=0.01,
)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: facebook/deit-small-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[inject_gating] 12개 레이어 중 12개에 gate 적용 (keep_cls_ungated=True)
cuda
NVIDIA GeForce RTX 5060 Ti


deit_small_gated_full_finetuning:   0%|          | 0/100 [00:00<?, ?it/s]

Test Loss=0.8143, Test Accuracy=0.7993
Epoch 000 | Train Loss=1.7680 | Train Acc=0.6353 | Test Loss=0.8143 | Test Acc=0.7993 | Gate Loss=0.00047
Test Loss=0.7714, Test Accuracy=0.8042
Epoch 001 | Train Loss=0.5394 | Train Acc=0.8609 | Test Loss=0.7714 | Test Acc=0.8042 | Gate Loss=0.00050
Test Loss=0.7725, Test Accuracy=0.8071
Epoch 002 | Train Loss=0.2816 | Train Acc=0.9238 | Test Loss=0.7725 | Test Acc=0.8071 | Gate Loss=0.00051
Test Loss=0.8902, Test Accuracy=0.7926
Epoch 003 | Train Loss=0.1626 | Train Acc=0.9549 | Test Loss=0.8902 | Test Acc=0.7926 | Gate Loss=0.00052
Test Loss=0.8989, Test Accuracy=0.7989
Epoch 004 | Train Loss=0.1184 | Train Acc=0.9661 | Test Loss=0.8989 | Test Acc=0.7989 | Gate Loss=0.00053
Test Loss=0.9743, Test Accuracy=0.7963
Epoch 005 | Train Loss=0.0981 | Train Acc=0.9713 | Test Loss=0.9743 | Test Acc=0.7963 | Gate Loss=0.00054
Early Stopping Triggered at epoch 5


2026/04/09 01:23:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/09 01:23:19 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


최고 성능 모델이 MLflow에 저장되었습니다.


In [5]:
import torch
from transformers import AutoModel
from src.model import dinosplus_classfier

device = "cuda" if torch.cuda.is_available() else "cpu"

backbone = AutoModel.from_pretrained("facebook/deit-small-patch16-224")
baseline_model = dinosplus_classfier(backbone, num=200)

baseline_model.load_state_dict(
    torch.load("/home/hyuksu/deep-learning-study/outputs/deit_small_baseline_full.pth", map_location=device)
)
baseline_model = baseline_model.to(device)
baseline_model.eval()

print("baseline 로드 완료!")

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: facebook/deit-small-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


baseline 로드 완료!


In [8]:
import torch
from transformers import AutoModel
from src.model import dinosplus_classfier
from src.re_make_Gate import inject_gating

device = "cuda" if torch.cuda.is_available() else "cpu"

backbone = AutoModel.from_pretrained("facebook/deit-small-patch16-224")
backbone, hooks = inject_gating(
    backbone,
    keep_cls_ungated=True,
    target_layers=None,
)

gated_model = dinosplus_classfier(backbone, num=200)

gated_model.load_state_dict(
    torch.load("/home/hyuksu/deep-learning-study/outputs/deit_small_gated_full.pth", map_location=device)
)
gated_model = gated_model.to(device)
gated_model.eval()

print("gated 로드 완료!")

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: facebook/deit-small-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[inject_gating] 12개 레이어 중 12개에 gate 적용 (keep_cls_ungated=True)
gated 로드 완료!


In [12]:
from src.not_gated_attention.run_analysis import run as run_baseline_attention
from datasets import load_dataset
import sys
sys.path.append("/home/hyuksu/deep-learning-study")
data = load_dataset("zh-plus/tiny-imagenet")
sample_img = data["valid"][0]["image"]

qk_maps = run_baseline_attention(
    model=baseline_model,
    pil_img=sample_img,
    save_dir="outputs/baseline_attention",
    img_size=224,
    patch_size=16,
    top_percent=10.0,
    layers=list(range(12)),   # 마지막 레이어 먼저 보는 걸 추천
)

ModuleNotFoundError: No module named 'src.attention_deep'

In [ ]:
from src.gated_attention.run_analysis import run as run_gated_attention
from datasets import load_dataset

data = load_dataset("zh-plus/tiny-imagenet")
sample_img = data["valid"][0]["image"]

qk_maps, gate_maps, results = run_gated_attention(
    model=gated_model,
    pil_img=sample_img,
    save_dir="outputs/gated_attention",
    img_size=224,
    patch_size=16,
    top_percent=10.0,
    layers=list(range(12)),   # 마지막 레이어 먼저
)

QK maps  : 12개 레이어
Gate maps: 12개 레이어
저장: outputs/gated_attention/floor12_overview.png

완료! 1개 레이어 → outputs/gated_attention/

  통계 분석 결과 (단일 gated 모델)

① Spearman  (gate평균 vs QK sink score)
   r=0.0839,  p=0.7954  ❌ 유의미하지않음
   → 양의 상관 or 무관

② Wilcoxon  (QK sink vs Gate×QK sink)
   stat=0.0000,  p=0.0005  ✅ 유의미한 억제

③ Kruskal-Wallis  stat=29.3379,  p=0.0020  ✅ 레이어간 차이 있음

④ Friedman  stat=6.7143,  p=0.2428  ❌ 차이 없음



In [ ]:
from src.gated_attention.run_analysis import run_comparison
from datasets import load_dataset

data = load_dataset("zh-plus/tiny-imagenet")
sample_img = data["valid"][0]["image"]

baseline_qk, gated_qk, gate_maps, results = run_comparison(
    baseline_model=baseline_model,
    gated_model=gated_model,
    pil_img=sample_img,
    save_dir="outputs/deit_compare",
    img_size=224,
    patch_size=16,
    top_percent=10.0,
    layers=[11],   # 일단 마지막 레이어부터
)